# 14. Supervised Learning: Logistic Regression

## Algorithm Category
**Type**: Supervised Learning - Classification  
**Complexity**: Low  
**Use Case**: Binary and multiclass classification

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of logistic regression
- Implement logistic regression using scikit-learn
- Evaluate classification performance using multiple metrics
- Interpret model coefficients and probabilities
- Apply logistic regression to real-world classification problems

## Historical Context

Logistic regression was developed in the 1950s and 1960s, building on earlier work in statistics. The logistic function (sigmoid) was first used by Pierre François Verhulst in the 19th century to model population growth. The application to binary classification was formalized by statisticians including David Cox.

**Key Papers/References:**
- Cox, D.R. (1958). "The regression analysis of binary sequences"
- Verhulst, P.F. (1838). "Notice sur la loi que la population suit dans son accroissement"

## When to Use Logistic Regression

Logistic regression is appropriate when:
- The target variable is categorical (binary or multiclass)
- You need probability estimates, not just class predictions
- Interpretability is important (coefficients show feature impact)
- The relationship between features and log-odds is approximately linear
- Dataset size is moderate (works well with thousands to millions of samples)

## Theory & Mechanics

### Mathematical Foundation

Logistic regression models the probability that a sample belongs to a particular class using the logistic (sigmoid) function:

**Binary Classification:**
$$P(y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_n x_n)}} = \sigma(z)$$

Where $\sigma(z)$ is the sigmoid function and $z = \beta_0 + \beta_1 x_1 + ... + \beta_n x_n$

**Log-Odds (Logit):**
$$\log\left(\frac{P(y=1|X)}{1-P(y=1|X)}\right) = \beta_0 + \beta_1 x_1 + ... + \beta_n x_n$$

### How It Works

1. **Objective**: Find coefficients that maximize the likelihood of observing the training data
2. **Cost Function**: Cross-entropy (log loss)
   $$L = -\frac{1}{n}\sum_{i=1}^{n}[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$$
3. **Optimization**: Typically uses gradient descent or Newton's method (no closed-form solution)
4. **Prediction**: Outputs probabilities, which are then thresholded (usually 0.5) to get class predictions

### Key Assumptions

1. **Linearity**: Log-odds are linear in features
2. **Independence**: Observations are independent
3. **No multicollinearity**: Features should not be highly correlated
4. **Large sample size**: Works better with more data

### Limitations

- Assumes linear decision boundary (can be extended with polynomial features)
- Sensitive to outliers
- Requires feature scaling for convergence
- May struggle with non-linear relationships


## Implementation

Let's implement logistic regression step by step using scikit-learn and our helper functions.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, cross_validate_model
from src.models.classification import (
    calculate_classification_metrics, plot_confusion_matrix,
    plot_roc_curve, plot_precision_recall_curve, analyze_confusion_matrix
)
from src.processing.preprocessing import scale_features
from src.utils.benchmarking import benchmark_model_training
from src.utils.traceability import extract_feature_importance_trace, save_traceability_data
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Load dataset - Breast Cancer (binary classification)
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='Target')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {cancer.target_names}")
print(f"Class distribution: {y.value_counts().to_dict()}")
print(f"\nFirst few rows:")
print(X.head())


In [ ]:
# Scale features (important for logistic regression)
X_scaled, scaler = scale_features(X, fit=True)

# Split data
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


In [ ]:
# Create and train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Intercept: {model.intercept_[0]:.3f}")
print(f"\nTop 5 Coefficients (by absolute value):")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
print(coef_df.head())


In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of positive class

# Evaluate model
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred, y_pred_proba)

print("Classification Metrics:")
print(f"  Accuracy: {metrics['accuracy']:.3f}")
print(f"  Precision: {metrics['precision']:.3f}")
print(f"  Recall: {metrics['recall']:.3f}")
print(f"  F1 Score: {metrics['f1_score']:.3f}")
if 'roc_auc' in metrics:
    print(f"  ROC-AUC: {metrics['roc_auc']:.3f}")


## Validation & Testing

Let's validate our model using multiple approaches: in-notebook assertions, cross-validation, and performance metrics.


In [ ]:
# Validation 1: Check model output validity
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
print("Model Output Validation:")
print(f"  Valid: {validation_result['valid']}")
if 'accuracy' in validation_result:
    print(f"  Accuracy: {validation_result['accuracy']:.3f}")
    print(f"  Number of classes: {validation_result['n_classes']}")

# Assertions
assert validation_result['valid'], "Model predictions are invalid!"
assert metrics['accuracy'] > 0.5, "Accuracy should be better than random!"
assert metrics['accuracy'] <= 1.0, "Accuracy cannot exceed 1.0"
print("\n✓ Basic validation checks passed")


In [ ]:
# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")
print(f"  Individual fold accuracies: {cv_scores}")

# Check cross-validation stability
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"\nCV Stability Check:")
print(f"  Coefficient of Variation: {stability['cv_coefficient']:.3f}")
print(f"  Is Stable: {stability['is_stable']}")

# Assertions
assert cv_mean > 0.5, "CV accuracy should be better than random!"
assert stability['is_stable'], "Cross-validation results are unstable!"
print("\n✓ Cross-validation checks passed")


In [ ]:
# Validation 3: Confusion matrix analysis
cm_analysis = analyze_confusion_matrix(y_test.values, y_pred)
print("Confusion Matrix Analysis:")
print(f"Confusion Matrix:\n{cm_analysis['confusion_matrix']}")
print(f"\nPer-Class Metrics:")
for class_name, class_metrics in cm_analysis['per_class_metrics'].items():
    print(f"  {class_name}: Precision={class_metrics['precision']:.3f}, "
          f"Recall={class_metrics['recall']:.3f}, F1={class_metrics['f1_score']:.3f}")

# Assertions
assert len(cm_analysis['per_class_metrics']) == 2, "Should have 2 classes for binary classification"
print("\n✓ Confusion matrix analysis complete")


## Performance Benchmarking

Let's benchmark the model's performance and compare with baseline.


In [ ]:
# Benchmark model training and prediction
benchmark_results = benchmark_model_training(
    LogisticRegression(max_iter=1000, random_state=42), 
    X_train.values, y_train.values, 
    X_test.values, y_test.values
)

print("Performance Benchmark:")
print(f"  Training Time: {benchmark_results['training_time']:.4f} seconds")
print(f"  Prediction Time: {benchmark_results['prediction_time']:.4f} seconds")
print(f"  Predictions per Second: {benchmark_results['predictions_per_second']:.0f}")
if 'test_accuracy' in benchmark_results:
    print(f"  Test Accuracy: {benchmark_results['test_accuracy']:.3f}")


## Traceability

Let's extract feature importance and create traceability records.


In [ ]:
# Extract feature importance (using coefficients)
feature_importance = extract_feature_importance_trace(model, feature_names=X.columns.tolist())
print("Feature Importance (by absolute coefficient):")
print(feature_importance.head(10))

# Visualize
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(10)
plt.barh(range(len(top_features)), top_features['importance'], align='center')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Absolute Coefficient Value')
plt.title('Top 10 Feature Importance (Logistic Regression)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Create traceability record
trace_data = {
    "model_type": "LogisticRegression",
    "dataset": "Breast Cancer",
    "n_samples": len(X),
    "n_features": X.shape[1],
    "coefficients": dict(zip(X.columns, model.coef_[0])),
    "intercept": float(model.intercept_[0]),
    "feature_importance": feature_importance.to_dict('records'),
    "performance_metrics": {
        "accuracy": float(metrics['accuracy']),
        "precision": float(metrics['precision']),
        "recall": float(metrics['recall']),
        "f1_score": float(metrics['f1_score']),
        "roc_auc": float(metrics.get('roc_auc', 0))
    },
    "cross_validation": {
        "mean_accuracy": float(cv_mean),
        "std_accuracy": float(cv_std)
    }
}

# Save traceability data
trace_path = save_traceability_data(trace_data, "logistic_regression_trace")
print(f"Traceability data saved to: {trace_path}")


## Visualization & Diagnostics

Let's visualize model performance with ROC curves, precision-recall curves, and confusion matrix.


In [ ]:
# Plot confusion matrix
plot_confusion_matrix(y_test.values, y_pred, 
                     class_names=cancer.target_names.tolist(),
                     title="Logistic Regression Confusion Matrix")


In [ ]:
# Plot ROC curve
roc_auc, fpr, tpr = plot_roc_curve(y_test.values, y_pred_proba, 
                                   title="Logistic Regression ROC Curve")
print(f"ROC-AUC Score: {roc_auc:.3f}")


In [ ]:
# Plot Precision-Recall curve
avg_precision, precision, recall = plot_precision_recall_curve(
    y_test.values, y_pred_proba, 
    title="Logistic Regression Precision-Recall Curve"
)
print(f"Average Precision: {avg_precision:.3f}")


## Real-World Application

Let's apply logistic regression to multiclass classification and hyperparameter tuning.


In [ ]:
# Multiclass example: Iris dataset
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = pd.Series(iris.target, name='Species')

X_iris_scaled, _ = scale_features(X_iris, fit=True)
X_iris_train, X_iris_test, y_iris_train, y_iris_test = split_data(
    X_iris_scaled, y_iris, test_size=0.2, random_state=42
)

# Train multiclass logistic regression
model_multi = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
model_multi.fit(X_iris_train, y_iris_train)

y_iris_pred = model_multi.predict(X_iris_test)
iris_accuracy = accuracy_score(y_iris_test, y_iris_pred)

print("Multiclass Classification (Iris Dataset):")
print(f"  Accuracy: {iris_accuracy:.3f}")
print(f"  Classes: {iris.target_names.tolist()}")

# Hyperparameter tuning
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2']}
grid_search = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42, solver='liblinear'),
                          param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Logistic Regression Basics**
   - Models probability of class membership using sigmoid function
   - Uses maximum likelihood estimation
   - Provides interpretable coefficients (log-odds)

2. **Model Evaluation**
   - Accuracy, precision, recall, F1-score for classification
   - ROC-AUC for binary classification performance
   - Confusion matrix for detailed error analysis

3. **Best Practices**
   - Scale features for convergence
   - Use cross-validation for robust evaluation
   - Tune regularization parameter (C)
   - Check for class imbalance

### When to Use Logistic Regression

✅ **Good for:**
- Binary and multiclass classification
- When probability estimates are needed
- Interpretability is important
- Baseline classifier for comparison
- Linear decision boundaries are sufficient

❌ **Not ideal for:**
- Non-linear decision boundaries (use kernel methods or neural networks)
- Very high-dimensional sparse data (use Naive Bayes)
- Complex feature interactions (use tree-based methods)

### Next Steps

- Try **Regularized Logistic Regression** (L1/L2) for feature selection
- Explore **Polynomial Features** for non-linear relationships
- Compare with **Support Vector Machines** for similar use cases
- Consider **Neural Networks** for complex non-linear patterns
